# 2.03 - Mental Health Dataset
## **Source Dataset:** 
### **mental_health_data.csv**

The 2023 BRFSS Survey provides detailed data about the mental health and physical health of individuals, specifically reporting on mental health days, poor health days, depressive disorders, and stress levels. It is particularly useful for analyzing the distribution and trends of mental health and associated factors across different states. This dataset includes information about the prevalence of mental health conditions and stress in the population, which can be valuable for research on public health and policy development.

* Source: Centers for Disease Control and Prevention (CDC), https://www.cdc.gov/brfss/annual_data/annual_2023.html
* MimeType: /text/csv

Columns:
* _STATE: A unique identifier for each state or region. The values are numeric FIPS codes tied to each state name.
  
* MENTHLTH: The number of days during the past 30 days that individuals reported their mental health as "not good". This is a **numeric value that can range from 1 to 30**, with special values:
    * 88: None (indicating no mental health issues).
    * 77: Don’t know.
    * 99: Refused.
    * Blank: Missing data.

* POORHLTH: The number of days during the past 30 days that poor physical or mental health kept individuals from performing their usual activities (e.g., work, self-care). This is a **numeric value that can range from 1 to 30**, with special values:
    * 88: None (indicating no mental health issues).
    * 77: Don’t know.
    * 99: Refused.
    * Blank: Missing data.

* ADDEPEV3: A binary indicator of whether an individual has ever been told by a healthcare provider that they have a depressive disorder (including major depression, dysthymia, or minor depression). The possible values are:
    * 1: Yes.
    * 2: No.
    * 7: Don’t know.
    * 9: Refused.
    * Blank: Missing data.

* SDHSTRE1 (not included due to too many NaN values): A measure of stress asking individuals how often they have felt stress in the last 30 days. The values are:
    * 1: Always.
    * 2: Usually.
    * 3: Sometimes.
    * 4: Rarely.
    * 5: Never.
    * 7: Don’t know.
    * 9: Refused.
    * Blank: Missing data.

# Features Added to mental_health_data.csv:
## **State-Based Features:**
* Average_Mental_Health_Days: The average number of days per 30 days individuals report poor mental health (MENTHLTH), aggregated at the state level.

* Average_Poor_Health_Days: The average number of days per 30 days poor health affected individuals' activities (POORHLTH), calculated by state.

* Depression_Prevalence: The proportion of people who report being diagnosed with a depressive disorder (ADDEPEV3), calculated for each state.

* Stress_Score (not icnuded due to too many NaN values): The average level of stress (on a scale of 1-5) experienced by individuals in each state, calculated from the responses to SDHSTRE1.

## **Handling Special Values:**
In the Mental Health Dataset, certain values are used to represent missing or non-standard responses. These special values were handled as follows:

* 88 (None): This value represents respondents who did not report poor mental health or poor health. For MENTHLTH and POORHLTH, this value was treated as 0, indicating no reported days of poor health.
* 77 (Don't know) and 99 (Refused): These values indicate uncertainty or refusal to answer. These responses were treated as missing (NaN) to ensure they do not skew the analysis of the average number of days with poor mental health, poor health affecting activities, depression prevalence, or stress levels.
* Blank (Missing Data): Any blank entries (i.e., missing responses) were also treated as missing (NaN). This allows for consistent calculations when determining averages or aggregating data by state.

# Get Datasets

In [18]:
import pandas as pd
import numpy as np

## Output CSV ##
outfile = "../data/processed/haunted_places_features_added.tab"

# Reading Haunted Places Dataset
haunted_places_df = pd.read_csv("../data/processed/haunted_places_cleaned.tab", sep="\t")

# Get the 2 datasets (Mental Health and Haunted Places)
mental_health_path = "../data/joined_datasets/mental_health_data.csv"

# Load datasets
mental_health_df = pd.read_csv(mental_health_path)

# Descriptive Feature Names from Mental Health Dataset
feature_names = [
    "Average_Mental_Health_Days",  # The average number of days individuals report poor mental health (MENTHLTH)
    "Average_Poor_Health_Days",  # The average number of days poor health affected activities (POORHLTH)
    "Depression_Prevalence",  # The proportion of people diagnosed with depression (ADDEPEV3)
]
print(mental_health_df.head())

   _STATE  FMONTH        IDATE IMONTH   IDAY    IYEAR  DISPCODE  \
0     1.0     1.0  b'03012023'  b'03'  b'01'  b'2023'    1100.0   
1     1.0     1.0  b'01062023'  b'01'  b'06'  b'2023'    1100.0   
2     1.0     1.0  b'03082023'  b'03'  b'08'  b'2023'    1100.0   
3     1.0     1.0  b'03062023'  b'03'  b'06'  b'2023'    1100.0   
4     1.0     1.0  b'01062023'  b'01'  b'06'  b'2023'    1100.0   

           SEQNO          _PSU  CTELENM1  ...  _RFBING6      _DRNKWK2  \
0  b'2023000001'  2.023000e+09       1.0  ...       1.0  5.397605e-79   
1  b'2023000002'  2.023000e+09       1.0  ...       1.0  5.397605e-79   
2  b'2023000003'  2.023000e+09       1.0  ...       1.0  5.397605e-79   
3  b'2023000004'  2.023000e+09       1.0  ...       1.0  5.397605e-79   
4  b'2023000005'  2.023000e+09       1.0  ...       1.0  4.700000e+01   

   _RFDRHV8  _FLSHOT7  _PNEUMO3  _AIDTST4  _RFSEAT2  _RFSEAT3  _DRNKDRV  state  
0       1.0       2.0       2.0       2.0       1.0       1.0       9.0    1.

# Mapping Labels

In [29]:
import pandas as pd
import numpy as np

# Preprocess the Mental Health Dataset

# Replace special values with NaN or 0
mental_health_df['MENTHLTH'] = mental_health_df['MENTHLTH'].replace({88: 0, 77: np.nan, 99: np.nan})
mental_health_df['POORHLTH'] = mental_health_df['POORHLTH'].replace({88: 0, 77: np.nan, 99: np.nan})
mental_health_df['ADDEPEV3'] = mental_health_df['ADDEPEV3'].replace({7: np.nan, 9: np.nan})

# Calculate state-level averages for MENTHLTH and POORHLTH
mental_health_state_avg = mental_health_df.groupby('_STATE').agg(
    Average_Mental_Health_Days=('MENTHLTH', 'mean'),
    Average_Poor_Health_Days=('POORHLTH', 'mean')
).reset_index()

# Round the new calculated features to 2 decimal places
mental_health_state_avg['Average_Mental_Health_Days'] = mental_health_state_avg['Average_Mental_Health_Days'].round(2)
mental_health_state_avg['Average_Poor_Health_Days'] = mental_health_state_avg['Average_Poor_Health_Days'].round(2)

# Calculate Depression Prevalence (proportion of '1' values in ADDEPEV3)
mental_health_state_avg['Depression_Prevalence'] = mental_health_df.groupby('_STATE')['ADDEPEV3'].apply(
    lambda x: (x == 1).sum() / ((x == 1).sum() + (x == 2).sum()) if ((x == 1).sum() + (x == 2).sum()) > 0 else np.nan
).reset_index(drop=True)

# Round the Depression Prevalence to 3 decimal places
mental_health_state_avg['Depression_Prevalence'] = mental_health_state_avg['Depression_Prevalence'].round(3)

# Create the state mapping dictionary (state code to state name)
state_code_to_name = {
    1: "Alabama", 2: "Alaska", 4: "Arizona", 5: "Arkansas", 6: "California",
    8: "Colorado", 9: "Connecticut", 10: "Delaware", 11: "District of Columbia", 12: "Florida",
    13: "Georgia", 15: "Hawaii", 16: "Idaho", 17: "Illinois", 18: "Indiana", 19: "Iowa", 
    20: "Kansas", 22: "Louisiana", 23: "Maine", 24: "Maryland", 25: "Massachusetts", 26: "Michigan", 
    27: "Minnesota", 28: "Mississippi", 29: "Missouri", 30: "Montana", 31: "Nebraska", 32: "Nevada", 
    33: "New Hampshire", 34: "New Jersey", 35: "New Mexico", 36: "New York", 37: "North Carolina", 
    38: "North Dakota", 39: "Ohio", 40: "Oklahoma", 41: "Oregon", 44: "Rhode Island", 45: "South Carolina", 
    46: "South Dakota", 47: "Tennessee", 48: "Texas", 49: "Utah", 50: "Vermont", 51: "Virginia", 
    53: "Washington", 54: "West Virginia", 55: "Wisconsin", 56: "Wyoming", 66: "Guam", 
    72: "Puerto Rico", 78: "Virgin Islands"
}

# Replace the _STATE values with state names
mental_health_state_avg['_STATE'] = mental_health_state_avg['_STATE'].map(state_code_to_name)

In [31]:
# Print the first few rows of the mental_health_state_avg DataFrame
# this dataframe gives all the possible values for the features for each state
print(mental_health_state_avg.head(100))

                  _STATE  Average_Mental_Health_Days  \
0                Alabama                        4.44   
1                 Alaska                        4.11   
2                Arizona                        4.06   
3               Arkansas                        4.74   
4             California                        4.63   
5               Colorado                        4.68   
6            Connecticut                        4.27   
7               Delaware                        4.06   
8   District of Columbia                        3.87   
9                Florida                        4.36   
10               Georgia                        4.30   
11                Hawaii                        4.07   
12                 Idaho                        4.51   
13              Illinois                        4.34   
14               Indiana                        4.61   
15                  Iowa                        4.51   
16                Kansas                        

# Merging

In [34]:
# Rename the '_STATE' column to 'State' to match the haunted_places_df column name
mental_health_state_avg.rename(columns={'_STATE': 'State'}, inplace=True)

# Merge datasets on 'State' column
merged_df = pd.merge(haunted_places_df, mental_health_state_avg, on="State", how="inner")

# Save dataset
print("Saving to CSV...")

# Read Feature added dataframe
out_df = pd.read_csv(f"{outfile}", sep="\t")

# Check if features exist and update them in out_df
for feature in feature_names:
    if feature in out_df.columns:
        # Update values from merged_df (not haunted_places_df)
        out_df[feature].update(merged_df[feature].values)
    else:
        # If the feature doesn't exist, add it from merged_df
        out_df[feature] = merged_df[feature]

# Save the updated dataset with the new features
out_df.to_csv(f"{outfile}", sep="\t", index=False)

print(f"CSV Saved to {outfile}")

# Display the merged dataframe with the mental health features
print(out_df[['State', 'Average_Mental_Health_Days', 'Average_Poor_Health_Days', 'Depression_Prevalence']].head(100))

Saving to CSV...
CSV Saved to ../data/processed/haunted_places_features_added.tab
       State  Average_Mental_Health_Days  Average_Poor_Health_Days  \
0   Michigan                        4.56                      5.72   
1   Michigan                        4.56                      5.72   
2   Michigan                        4.56                      5.72   
3   Michigan                        4.56                      5.72   
4   Michigan                        4.56                      5.72   
..       ...                         ...                       ...   
95  Michigan                        4.56                      5.72   
96  Michigan                        4.56                      5.72   
97  Michigan                        4.56                      5.72   
98  Michigan                        4.56                      5.72   
99  Michigan                        4.56                      5.72   

    Depression_Prevalence  
0                   0.234  
1                   0